In [ ]:
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "mesale/gpt2-from-scratch"
!git clone https://{token}@github.com/{REPO}.git /kaggle/temp/repo
%cd /kaggle/temp/repo
!git remote set-url origin https://github.com/{REPO}.git
!pip install -q tokenizers pyyaml

In [ ]:
import glob
bins = glob.glob("/kaggle/input/**/tinystories_train.bin", recursive=True)
assert bins, "tinystories_train.bin not found: is the dataset attached, and did data-prep finish?"
DATA_DIR = os.path.dirname(bins[0])
print(DATA_DIR, os.listdir(DATA_DIR))
n = os.path.getsize(bins[0]) // 2          # uint16 = 2 bytes per token
print(f"{n:,} train tokens | the run sees {8000 * 32768:,} tokens = {8000 * 32768 / n:.2f} epochs")

In [ ]:
!python -m src.train --config configs/tinystories_main.yaml --set train.data_dir={DATA_DIR} train.out_dir=/kaggle/temp/smoke train.max_steps=60 train.eval_interval=30 train.eval_iters=5 train.log_interval=10 train.warmup_steps=10

In [ ]:
prev = glob.glob("/kaggle/input/**/tinystories_main/ckpt.pt", recursive=True)
if prev:
    os.makedirs("/kaggle/working/runs/tinystories_main", exist_ok=True)
    !cp {prev[0]} /kaggle/working/runs/tinystories_main/ckpt.pt

In [ ]:
!python -m src.train --config configs/tinystories_main.yaml --set train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs

In [ ]:
import shutil, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, pandas as pd
from src.checkpoint import load_model
from src.tokenizer import load_tokenizer
from src.generate import sample_texts

run, res = "/kaggle/working/runs/tinystories_main", "/kaggle/temp/repo/results/tinystories_main"
os.makedirs(res, exist_ok=True)
tr, ev = pd.read_csv(f"{run}/train_log.csv"), pd.read_csv(f"{run}/eval_log.csv")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tr.step, tr.loss, alpha=0.6, label="train (running avg)")
ax.plot(ev.step, ev.val_loss, "o-", label="val")
ax.set_xlabel("step"); ax.set_ylabel("loss (nats/token)"); ax.set_ylim(top=4); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f"{res}/loss_curves.png", dpi=150, bbox_inches="tight")

model, tok = load_model(f"{run}/best.pt", "cuda"), load_tokenizer("configs/tokenizer.json")
with open(f"{res}/samples.txt", "w") as f:
    for temp in [0, 0.7, 1.0]:
        for prompt in ["Once upon a time", "Tom and Lily went to the park.", "The little dog was sad because"]:
            for t in sample_texts(model, tok, prompt, n=1 if temp == 0 else 2, max_new_tokens=200,
                                  temperature=temp, seed=0, device="cuda"):
                f.write(f"### temperature={temp} | prompt={prompt!r}\n{t}\n\n")
for fn in ["train_log.csv", "eval_log.csv", "config.yaml"]:
    shutil.copy(f"{run}/{fn}", res)

In [ ]:
%cd /kaggle/temp/repo
!git config user.email "mesalemtsegaye@gmail.com" && git config user.name "kaggle"
!git add results && git commit -m "Add TinyStories main run results" && git push https://{token}@github.com/{REPO}.git HEAD:main